# Module 8 — Lab 4: RAG Evaluation Lab
## Retrieval Metrics (Hit Rate, MRR, Precision@K, Recall@K, nDCG) & End-to-End Metrics (Answer Relevance, Groundedness, Hallucination Rate, Coherence)

---

### Lab Overview

Building a RAG system is one thing — **knowing whether it actually works** is another. In production, a poorly evaluated RAG pipeline can silently serve hallucinated answers, miss critical documents, or return irrelevant context. This lab teaches you to **measure what matters** using custom-built evaluation functions (no framework dependencies).

We evaluate at **two levels**:

| Level | What We Measure | Metrics |
|-------|----------------|---------|
| **Retrieval** | Did the retriever find the right chunks? | Hit Rate, MRR, Precision@K, Recall@K, nDCG@K |
| **End-to-End (Generation)** | Did the LLM produce a good answer from context? | Answer Relevance, Groundedness, Hallucination Rate, Coherence |

We use a **curated evaluation dataset** of 40 QA pairs with both `ground_truth` (ideal answers) and `gold_context` (verbatim source passages) — enabling rigorous evaluation at both levels.

**Key Design:** All configurable RAG pipeline parameters are defined in a single **Configuration Cell** (Section 3). To re-run with different settings, just change values there and hit **Runtime → Run All**. Results are exported to Excel at the end for experiment tracking.

| Step | What We Do | Key Technology |
|------|-----------|----------------|
| 1 | Install dependencies | `langchain`, `faiss-cpu`, `openai`, `openpyxl` |
| 2 | API key configuration | Google Colab Secrets Manager |
| 3 | **Configuration cell** — all tunable RAG parameters in one place | Python constants |
| 4 | Load eval dataset + build RAG pipeline using config | FAISS, LCEL chain |
| 5 | Implement 5 custom retrieval metrics | Pure Python + NumPy |
| 6 | Run retrieval evaluation (aggregate + per-paper + per-query) | Batch analysis |
| 7 | K-value sensitivity analysis | Sweep K=1→10 |
| 8 | Implement 4 custom end-to-end metrics (LLM-as-judge) | `gpt-4.1-mini` as evaluator |
| 9 | Run end-to-end evaluation across 40 queries | Batch scoring |
| 10 | Error analysis + combined dashboard | Diagnostics |
| 11 | **Export all results to Excel** with config snapshot | `openpyxl` |

### Learning Objectives

By the end of this lab, you will be able to:
1. Implement retrieval metrics (Hit Rate, MRR, Precision@K, Recall@K, nDCG@K) from scratch
2. Understand the mathematical formula behind each metric and when to use it
3. Build LLM-as-judge evaluators for answer relevance, groundedness, hallucination rate, and coherence
4. Run a full evaluation pipeline over a curated 40-question dataset
5. Design reproducible experiment configurations for A/B testing RAG parameter changes
6. Export evaluation results to Excel for cross-experiment comparison

---

## 1. Install Dependencies

In [ ]:
# Core LangChain + OpenAI
!pip install langchain langchain-openai langchain-community -q

# FAISS vector search + PDF/JSON processing
!pip install faiss-cpu pymupdf jq -q

# For downloading dataset
!pip install gdown -q

# NumPy for metrics, openpyxl for Excel export
!pip install numpy openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 35.6 MB/s eta 0:00:00


## 2. API Key Configuration

> **Setup:** Go to 🔑 icon in Colab's left sidebar → Add `OPENAI_API_KEY` → Toggle "Notebook access" ON.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

status = "✅ Loaded" if os.environ.get("OPENAI_API_KEY") else "❌ MISSING"
print(f"OPENAI_API_KEY: {status}")

OPENAI_API_KEY: ✅ Loaded


## 3. ⚙️ RAG Pipeline Configuration — Single Control Panel

**This is the only cell you need to change to re-run the entire evaluation with different settings.**

After changing any value below, simply go to **Runtime → Run All** to rebuild the pipeline and re-evaluate with the new configuration. All results (including these settings) are exported to Excel at the end for experiment tracking.

> **Business Scenario:** A Bajaj Allianz data science team runs this notebook weekly with different configurations — comparing `text-embedding-3-small` vs `text-embedding-3-large`, or `chunk_size=2000` vs `chunk_size=4000` — and tracks results in a shared Excel sheet to converge on optimal production settings.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║           ⚙️  RAG PIPELINE CONFIGURATION                       ║
# ║  Change values here, then Runtime → Run All to re-evaluate     ║
# ╚══════════════════════════════════════════════════════════════════╝

# --- Embedding Model ---
EMBEDDING_MODEL = "text-embedding-3-small"   # Options: "text-embedding-3-small" (1536d), "text-embedding-3-large" (3072d)

# --- Generator LLM ---
GENERATOR_LLM = "gpt-4.1-mini"              # Options: "gpt-4.1-mini", "gpt-5-mini" (reasoning, no temp), etc.
GENERATOR_TEMPERATURE = 0                    # 0 = deterministic, 0.7 = creative (only for non-reasoning models)

# --- Evaluator / Judge LLM ---
JUDGE_LLM = "gpt-5.2"                  # LLM used for scoring answer quality (LLM-as-judge)
JUDGE_TEMPERATURE = 0                        # Keep at 0 for consistent scoring

# --- Document Chunking ---
CHUNK_SIZE = 3500                            # Max characters per chunk (try: 1000, 2000, 3500, 5000)
CHUNK_OVERLAP = 500                          # Overlap between consecutive chunks (try: 0, 100, 200, 500)

# --- Retrieval ---
SEARCH_TYPE = "similarity"                   # Options: "similarity", "mmr"
TOP_K = 5                                    # Number of chunks to retrieve (try: 3, 5, 7, 10)
MMR_FETCH_K = 20                             # Only used if SEARCH_TYPE="mmr": candidate pool size
MMR_LAMBDA = 0.5                             # Only used if SEARCH_TYPE="mmr": diversity factor (0=diverse, 1=relevant)

# --- Relevance Matching (for retrieval evaluation) ---
RELEVANCE_THRESHOLD = 0.3                    # Word overlap threshold to consider a chunk "relevant" (0.0-1.0)

# --- Experiment Metadata ---
EXPERIMENT_NAME = "baseline_v1"              # A label for this run (appears in the Excel export)
EXPERIMENT_NOTES = "Default settings, FAISS similarity top-5, chunk_size=3500"

# --- Output ---
EXCEL_OUTPUT_PATH = f"rag_eval_results_{EXPERIMENT_NAME}.xlsx"

# ─────────────────────────────────────────────────────────────
# Print config summary
print("⚙️  RAG PIPELINE CONFIGURATION")
print("=" * 55)
config_items = {
    "Embedding Model": EMBEDDING_MODEL,
    "Generator LLM": GENERATOR_LLM,
    "Generator Temperature": GENERATOR_TEMPERATURE,
    "Judge LLM": JUDGE_LLM,
    "Chunk Size": f"{CHUNK_SIZE} chars",
    "Chunk Overlap": f"{CHUNK_OVERLAP} chars",
    "Search Type": SEARCH_TYPE,
    "Top-K": TOP_K,
    "Relevance Threshold": RELEVANCE_THRESHOLD,
    "Experiment": EXPERIMENT_NAME,
}
for key, val in config_items.items():
    print(f"  {key:<25s} → {val}")
print(f"\n📁 Results will be saved to: {EXCEL_OUTPUT_PATH}")

⚙️  RAG PIPELINE CONFIGURATION
  Embedding Model           → text-embedding-3-small
  Generator LLM             → gpt-4.1-mini
  Generator Temperature     → 0
  Judge LLM                 → gpt-5.2
  Chunk Size                → 3500 chars
  Chunk Overlap             → 500 chars
  Search Type               → similarity
  Top-K                     → 5
  Relevance Threshold       → 0.3
  Experiment                → baseline_v1

📁 Results will be saved to: rag_eval_results_baseline_v1.xlsx


## 4. Load Evaluation Dataset & Build RAG Pipeline (Using Config)

We load the 40-question eval dataset and build the RAG pipeline using **all parameters from the config cell above**.

In [ ]:
import json
import numpy as np
from datetime import datetime

# --- Upload the eval dataset ---
from google.colab import files
print("📤 Upload rag_eval_dataset_40_questions.json:")
uploaded = files.upload()

eval_file = list(uploaded.keys())[0]
with open(eval_file, 'r') as f:
    eval_dataset = json.load(f)

print(f"\n✅ Loaded {len(eval_dataset)} evaluation QA pairs")

# Record the run timestamp
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"⏱️  Run started: {RUN_TIMESTAMP}")

📤 Upload rag_eval_dataset_40_questions.json:


Saving rag_eval_dataset_40_questions.json to rag_eval_dataset_40_questions.json

✅ Loaded 40 evaluation QA pairs
⏱️  Run started: 2026-03-18 13:03:21


In [1]:
# --- Download and process the document corpus ---
!gdown 1aZxZejfteVuofISodUrY2CDoyuPLYDGZ
!unzip -o rag_docs.zip

Downloading...
From: https://drive.google.com/uc?id=1aZxZejfteVuofISodUrY2CDoyuPLYDGZ
To: /content/rag_docs.zip
100% 5.92M/5.92M [00:00<00:00, 25.2MB/s]
Archive:  rag_docs.zip
   creating: rag_docs/
  inflating: rag_docs/attention_paper.pdf  
  inflating: rag_docs/cnn_paper.pdf  
  inflating: rag_docs/resnet_paper.pdf  
  inflating: rag_docs/vision_transformer.pdf  
  inflating: rag_docs/wikidata_rag_demo.jsonl  


In [ ]:
import json
from glob import glob
from langchain_community.document_loaders import JSONLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # Corrected import path
from langchain_core.documents import Document # Corrected import path for Document

# # Load Wikipedia articles
# loader = JSONLoader(file_path='./rag_docs/wikidata_rag_demo.jsonl',
#                     jq_schema='.', text_content=False, json_lines=True)
# wiki_raw = loader.load()
# wiki_docs = []
# for doc in wiki_raw:
#     data = json.loads(doc.page_content)
#     content = ' '.join(data['paragraphs'])
#     wiki_docs.append(Document(page_content=content,
#                               metadata={"title": data['title'], "id": data['id'], "source": "Wikipedia"}))

# Load and chunk PDFs — USING CONFIG VALUES
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,        # ← from config
    chunk_overlap=CHUNK_OVERLAP   # ← from config
)
pdf_files = sorted(glob('./rag_docs/*.pdf'))
paper_docs = []
for fp in pdf_files:
    pages = PyMuPDFLoader(fp).load()
    paper_docs.extend(splitter.split_documents(pages))

total_docs = paper_docs.copy()
# print(f"✅ Corpus: {len(wiki_docs)} wiki + {len(paper_docs)} PDF chunks = {len(total_docs)} total")
print(f"   (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")
print(len(total_docs))

   (chunk_size=3500, chunk_overlap=500)
79


In [ ]:
# --- Build FAISS index — USING CONFIG EMBEDDING MODEL ---
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embed_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)  # ← from config
faiss_db = FAISS.from_documents(documents=total_docs, embedding=embed_model)
print(f"✅ FAISS index: {faiss_db.index.ntotal} vectors (model: {EMBEDDING_MODEL})")

# --- Build retriever — USING CONFIG SEARCH PARAMS ---
retriever_kwargs = {"k": TOP_K}
if SEARCH_TYPE == "mmr":
    retriever_kwargs["fetch_k"] = MMR_FETCH_K
    retriever_kwargs["lambda_mult"] = MMR_LAMBDA

retriever = faiss_db.as_retriever(
    search_type=SEARCH_TYPE,      # ← from config
    search_kwargs=retriever_kwargs
)
print(f"✅ Retriever: {SEARCH_TYPE}, top-{TOP_K}")

# --- Build RAG chain — USING CONFIG LLM ---
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

generator_llm = ChatOpenAI(model_name=GENERATOR_LLM, temperature=GENERATOR_TEMPERATURE)  # ← from config
judge_llm = ChatOpenAI(model_name=JUDGE_LLM, temperature=JUDGE_TEMPERATURE)              # ← from config

rag_prompt = ChatPromptTemplate.from_template(
    """You are an expert assistant. Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't know." Keep the answer detailed.

Question: {question}
Context: {context}
Answer:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | generator_llm
)
print(f"✅ RAG chain ready (generator: {GENERATOR_LLM}, judge: {JUDGE_LLM})")

✅ FAISS index: 79 vectors (model: text-embedding-3-small)
✅ Retriever: similarity, top-5
✅ RAG chain ready (generator: gpt-4.1-mini, judge: gpt-5.2)


## 5. Retrieval Metrics — Theory & Custom Implementation

### 5.1 Hit Rate@K
$$\text{Hit Rate} = \frac{\text{Queries with} \geq 1 \text{ relevant doc in top-K}}{\text{Total queries}}$$
Quick sanity check — if this is low, the retriever is fundamentally broken. **Range:** 0→1

### 5.2 Mean Reciprocal Rank (MRR@K)
$$\text{MRR} = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{\text{rank}_i}$$
How **early** does the first relevant doc appear? **Range:** 0→1 (1.0 = first result always relevant)

### 5.3 Precision@K
$$\text{Precision@K} = \frac{\text{Relevant docs in top-K}}{K}$$
Of the K docs retrieved, how many are relevant? Measures **noise** in context. **Range:** 0→1

### 5.4 Recall@K
$$\text{Recall@K} = \frac{\text{Relevant docs in top-K}}{\text{Total relevant docs}}$$
Of all relevant passages, how many did we find? Measures **completeness**. **Range:** 0→1

### 5.5 nDCG@K (Normalized Discounted Cumulative Gain)
$$\text{DCG@K} = \sum_{i=1}^{K} \frac{\text{rel}_i}{\log_2(i+1)}, \quad \text{nDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$
Are the **most relevant** docs ranked **highest**? Penalizes burying good docs lower. **Range:** 0→1

In [ ]:
# ============================================================
# CUSTOM RETRIEVAL METRIC FUNCTIONS — No frameworks, pure Python
# ============================================================

def is_relevant(retrieved_text, gold_context, threshold=RELEVANCE_THRESHOLD):
    """Check if a retrieved chunk is relevant via word overlap with gold context."""
    gold_words = set(gold_context.lower().split())
    retrieved_words = set(retrieved_text.lower().split())
    if not gold_words:
        return False
    overlap = len(gold_words & retrieved_words) / len(gold_words)
    return overlap >= threshold


def hit_rate_at_k(retrieved_docs_list, gold_contexts, k=TOP_K):
    """Hit Rate@K: fraction of queries with at least one relevant doc in top-K."""
    hits = 0
    for docs, gold in zip(retrieved_docs_list, gold_contexts):
        if any(is_relevant(d.page_content, gold) for d in docs[:k]):
            hits += 1
    return hits / len(gold_contexts)


def mrr_at_k(retrieved_docs_list, gold_contexts, k=TOP_K):
    """Mean Reciprocal Rank@K: average of 1/rank of first relevant doc."""
    rrs = []
    for docs, gold in zip(retrieved_docs_list, gold_contexts):
        rr = 0.0
        for rank, d in enumerate(docs[:k], start=1):
            if is_relevant(d.page_content, gold):
                rr = 1.0 / rank
                break
        rrs.append(rr)
    return np.mean(rrs)


def precision_at_k(retrieved_docs_list, gold_contexts, k=TOP_K):
    """Precision@K: fraction of top-K docs that are relevant, averaged across queries."""
    precs = []
    for docs, gold in zip(retrieved_docs_list, gold_contexts):
        rel_count = sum(1 for d in docs[:k] if is_relevant(d.page_content, gold))
        precs.append(rel_count / k)
    return np.mean(precs)


def recall_at_k(retrieved_docs_list, gold_contexts, k=TOP_K):
    """Recall@K: binary per query (1 if any relevant doc found, 0 otherwise)."""
    recs = []
    for docs, gold in zip(retrieved_docs_list, gold_contexts):
        found = any(is_relevant(d.page_content, gold) for d in docs[:k])
        recs.append(1.0 if found else 0.0)
    return np.mean(recs)


def ndcg_at_k(retrieved_docs_list, gold_contexts, k=TOP_K):
    """nDCG@K: normalized discounted cumulative gain (binary relevance)."""
    ndcgs = []
    for docs, gold in zip(retrieved_docs_list, gold_contexts):
        rels = [1.0 if is_relevant(d.page_content, gold) else 0.0 for d in docs[:k]]
        dcg = sum(r / np.log2(i + 2) for i, r in enumerate(rels))
        ideal_rels = sorted(rels, reverse=True)
        idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal_rels))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    return np.mean(ndcgs)


print("✅ 5 retrieval metric functions defined (using config: threshold={}, k={})".format(
    RELEVANCE_THRESHOLD, TOP_K))

✅ 5 retrieval metric functions defined (using config: threshold=0.3, k=5)


## 6. Run Retrieval Evaluation Across 40 Queries

In [ ]:
# Step 1: Retrieve documents for all 40 queries
all_retrieved = []
all_gold_contexts = []

print(f"Retrieving top-{TOP_K} documents for 40 queries ({SEARCH_TYPE} search)...")
for item in eval_dataset:
    docs = retriever.invoke(item["question"])
    all_retrieved.append(docs)
    all_gold_contexts.append(item["gold_context"])

print(f"✅ Retrieval complete for {len(all_retrieved)} queries")

# Step 2: Compute all retrieval metrics
retrieval_metrics = {
    f"Hit Rate@{TOP_K}": hit_rate_at_k(all_retrieved, all_gold_contexts, TOP_K),
    f"MRR@{TOP_K}": mrr_at_k(all_retrieved, all_gold_contexts, TOP_K),
    f"Precision@{TOP_K}": precision_at_k(all_retrieved, all_gold_contexts, TOP_K),
    f"Recall@{TOP_K}": recall_at_k(all_retrieved, all_gold_contexts, TOP_K),
    f"nDCG@{TOP_K}": ndcg_at_k(all_retrieved, all_gold_contexts, TOP_K),
}

print(f"\n📊 RETRIEVAL METRICS (K={TOP_K}, search={SEARCH_TYPE})")
print("=" * 55)
for name, value in retrieval_metrics.items():
    bar = "█" * int(value * 30) + "░" * (30 - int(value * 30))
    print(f"  {name:<18s} {value:.4f}  {bar}")

Retrieving top-5 documents for 40 queries (similarity search)...
✅ Retrieval complete for 40 queries

📊 RETRIEVAL METRICS (K=5, search=similarity)
  Hit Rate@5         1.0000  ██████████████████████████████
  MRR@5              0.9271  ███████████████████████████░░░
  Precision@5        0.8300  ████████████████████████░░░░░░
  Recall@5           1.0000  ██████████████████████████████
  nDCG@5             0.9382  ████████████████████████████░░


In [ ]:
# Per-paper breakdown
from collections import defaultdict

paper_retrieval = defaultdict(lambda: {"retrieved": [], "gold": []})
for item, docs in zip(eval_dataset, all_retrieved):
    paper_retrieval[item["source_document"]]["retrieved"].append(docs)
    paper_retrieval[item["source_document"]]["gold"].append(item["gold_context"])

print(f"📊 RETRIEVAL METRICS — PER PAPER (K={TOP_K})")
print("=" * 85)
print(f"  {'Paper':<30s} {'Hit':>6s} {'MRR':>6s} {'P@K':>6s} {'R@K':>6s} {'nDCG':>6s}")
print("-" * 85)

paper_ret_detail = {}
for paper in sorted(paper_retrieval.keys()):
    r = paper_retrieval[paper]
    vals = {
        "hit": hit_rate_at_k(r["retrieved"], r["gold"], TOP_K),
        "mrr": mrr_at_k(r["retrieved"], r["gold"], TOP_K),
        "prec": precision_at_k(r["retrieved"], r["gold"], TOP_K),
        "rec": recall_at_k(r["retrieved"], r["gold"], TOP_K),
        "ndcg": ndcg_at_k(r["retrieved"], r["gold"], TOP_K),
    }
    paper_ret_detail[paper] = vals
    name = paper.replace('.pdf', '').replace('_', ' ')[:28]
    print(f"  {name:<30s} {vals['hit']:>6.3f} {vals['mrr']:>6.3f} {vals['prec']:>6.3f} {vals['rec']:>6.3f} {vals['ndcg']:>6.3f}")

📊 RETRIEVAL METRICS — PER PAPER (K=5)
  Paper                             Hit    MRR    P@K    R@K   nDCG
-------------------------------------------------------------------------------------
  attention paper                 1.000  0.950  0.660  1.000  0.931
  cnn paper                       1.000  1.000  0.940  1.000  0.995
  resnet paper                    1.000  0.883  0.900  1.000  0.926
  vision transformer              1.000  0.875  0.820  1.000  0.901


In [ ]:
# Per-query detail with relevance status
per_query_retrieval = []
for item, docs, gold in zip(eval_dataset, all_retrieved, all_gold_contexts):
    hit = 1 if any(is_relevant(d.page_content, gold) for d in docs[:TOP_K]) else 0
    rr = 0.0
    for rank, d in enumerate(docs[:TOP_K], start=1):
        if is_relevant(d.page_content, gold):
            rr = 1.0 / rank
            break
    p_k = sum(1 for d in docs[:TOP_K] if is_relevant(d.page_content, gold)) / TOP_K
    per_query_retrieval.append({
        "question_id": item["question_id"],
        "source": item["source_document"],
        "question": item["question"],
        "hit": hit, "reciprocal_rank": rr, "precision_at_k": p_k
    })

# Show misses
misses = [q for q in per_query_retrieval if q["hit"] == 0]
print(f"\n❌ Retrieval misses: {len(misses)}/{len(per_query_retrieval)} queries")
for m in misses:
    print(f"   {m['question_id']:<8s} {m['question'][:65]}")


❌ Retrieval misses: 0/40 queries


## 7. K-Value Sensitivity Analysis

How do metrics change as we increase K from 1 to 10? Helps decide optimal retrieval depth.

In [ ]:
k_values = [1, 2, 3, 5, 7, 10]

print(f"{'K':>3s} {'Hit Rate':>10s} {'MRR':>8s} {'P@K':>8s} {'R@K':>8s} {'nDCG@K':>8s}")
print("-" * 50)

k_sweep_results = []
for k in k_values:
    row = {
        "k": k,
        "hit_rate": hit_rate_at_k(all_retrieved, all_gold_contexts, k),
        "mrr": mrr_at_k(all_retrieved, all_gold_contexts, k),
        "precision": precision_at_k(all_retrieved, all_gold_contexts, k),
        "recall": recall_at_k(all_retrieved, all_gold_contexts, k),
        "ndcg": ndcg_at_k(all_retrieved, all_gold_contexts, k),
    }
    k_sweep_results.append(row)
    print(f"{k:>3d} {row['hit_rate']:>10.3f} {row['mrr']:>8.3f} {row['precision']:>8.3f} "
          f"{row['recall']:>8.3f} {row['ndcg']:>8.3f}")

print("\n💡 Hit Rate/Recall increase with K; Precision typically decreases; MRR is K-independent.")

  K   Hit Rate      MRR      P@K      R@K   nDCG@K
--------------------------------------------------
  1      0.875    0.875    0.875    0.875    0.875
  2      0.950    0.912    0.838    0.950    0.922
  3      0.975    0.921    0.842    0.975    0.927
  5      1.000    0.927    0.830    1.000    0.938
  7      1.000    0.927    0.593    1.000    0.938
 10      1.000    0.927    0.415    1.000    0.938

💡 Hit Rate/Recall increase with K; Precision typically decreases; MRR is K-independent.


## 8. End-to-End Metrics — Theory & LLM-as-Judge Implementation

| Metric | Scale | What It Measures |
|--------|-------|-----------------|
| **Answer Relevance** | 1-5 | Does the answer address the question? |
| **Groundedness** | 1-5 | Is every claim supported by the retrieved context? |
| **Hallucination Rate** | 0.0-1.0 | Fraction of claims NOT supported by context |
| **Coherence** | 1-5 | Logical flow, structure, readability |

$$\text{Hallucination Rate} = \frac{\text{Unsupported claims}}{\text{Total claims in answer}}$$

In [ ]:
import json as json_lib

def llm_judge(prompt, llm):
    """Send a judging prompt to the LLM and parse JSON response."""
    result = llm.invoke(prompt)
    text = result.content.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        return json_lib.loads(text)
    except:
        return None

def evaluate_answer_relevance(question, answer, llm):
    prompt = f"""Rate how well the answer addresses the question (1-5).
1=Completely irrelevant  2=Partially relevant  3=Moderate  4=Highly relevant  5=Perfect

Question: {question}
Answer: {answer}

Return ONLY JSON: {{"score": <int>, "reason": "<brief>"}}"""
    return llm_judge(prompt, llm)

def evaluate_groundedness(context, answer, llm):
    prompt = f"""Check if every claim in the answer is supported by the context (1-5).
1=Unsupported  2=Poorly supported  3=Partially  4=Well supported  5=Fully supported

Context: {context[:3000]}
Answer: {answer}

Return ONLY JSON: {{"score": <int>, "reason": "<brief>"}}"""
    return llm_judge(prompt, llm)

def evaluate_hallucination_rate(context, answer, llm):
    prompt = f"""Count factual claims in the answer. Check each against context.

Context: {context[:3000]}
Answer: {answer}

Return ONLY JSON: {{"total_claims": <int>, "unsupported_claims": <int>, "hallucination_rate": <float 0.0-1.0>, "reason": "<brief>"}}"""
    return llm_judge(prompt, llm)

def evaluate_coherence(answer, llm):
    prompt = f"""Rate the coherence: logical flow, organization, clarity (1-5).
1=Incoherent  2=Poor  3=Adequate  4=Well-structured  5=Excellent

Answer: {answer}

Return ONLY JSON: {{"score": <int>, "reason": "<brief>"}}"""
    return llm_judge(prompt, llm)

print(f"✅ 4 LLM-as-judge functions defined (judge model: {JUDGE_LLM})")

✅ 4 LLM-as-judge functions defined (judge model: gpt-5.2)


## 9. Run End-to-End Evaluation Across 40 Queries

> **Business Scenario:** Apollo Hospitals runs weekly RAG evaluations. If groundedness drops below 4.0 or hallucination rate exceeds 0.15, the clinical Q&A system is paused for review.

In [ ]:
e2e_results = []

print(f"Running end-to-end evaluation ({GENERATOR_LLM} + {JUDGE_LLM})...")
print("(~3-5 minutes for 40 queries × 5 LLM calls each)\n")

for i, item in enumerate(eval_dataset):
    query = item["question"]

    # Retrieve + Generate
    retrieved_docs = retriever.invoke(query)
    context = format_docs(retrieved_docs)
    rag_answer = rag_chain.invoke(query).content

    # Evaluate with LLM-as-judge
    rel = evaluate_answer_relevance(query, rag_answer, judge_llm)
    grd = evaluate_groundedness(context, rag_answer, judge_llm)
    hal = evaluate_hallucination_rate(context, rag_answer, judge_llm)
    coh = evaluate_coherence(rag_answer, judge_llm)

    result = {
        "question_id": item["question_id"],
        "source": item["source_document"],
        "question": item["question"],
        "rag_answer": rag_answer[:500],
        "answer_relevance": rel.get("score", 0) if rel else 0,
        "groundedness": grd.get("score", 0) if grd else 0,
        "hallucination_rate": hal.get("hallucination_rate", -1) if hal else -1,
        "coherence": coh.get("score", 0) if coh else 0,
        "rel_reason": (rel or {}).get("reason", ""),
        "grd_reason": (grd or {}).get("reason", ""),
    }

    # Also attach retrieval hit info
    gold = item["gold_context"]
    result["retrieval_hit"] = 1 if any(is_relevant(d.page_content, gold) for d in retrieved_docs[:TOP_K]) else 0

    e2e_results.append(result)

    if (i + 1) % 10 == 0:
        print(f"  Completed {i+1}/40 queries...")

print(f"\n✅ Evaluation complete for {len(e2e_results)} queries")

Running end-to-end evaluation (gpt-4.1-mini + gpt-5.2)...
(~3-5 minutes for 40 queries × 5 LLM calls each)

  Completed 10/40 queries...
  Completed 20/40 queries...
  Completed 30/40 queries...
  Completed 40/40 queries...

✅ Evaluation complete for 40 queries


In [ ]:
# Aggregate generation metrics
rel_scores = [r["answer_relevance"] for r in e2e_results if r["answer_relevance"] > 0]
grd_scores = [r["groundedness"] for r in e2e_results if r["groundedness"] > 0]
hal_rates = [r["hallucination_rate"] for r in e2e_results if r["hallucination_rate"] >= 0]
coh_scores = [r["coherence"] for r in e2e_results if r["coherence"] > 0]

generation_metrics = {
    "Answer Relevance (1-5)": np.mean(rel_scores) if rel_scores else 0,
    "Groundedness (1-5)": np.mean(grd_scores) if grd_scores else 0,
    "Hallucination Rate (0-1)": np.mean(hal_rates) if hal_rates else 0,
    "Coherence (1-5)": np.mean(coh_scores) if coh_scores else 0,
}

print("📊 END-TO-END METRICS")
print("=" * 55)
for name, val in generation_metrics.items():
    print(f"  {name:<28s} {val:.4f}")

📊 END-TO-END METRICS
  Answer Relevance (1-5)       4.8750
  Groundedness (1-5)           3.3750
  Hallucination Rate (0-1)     0.4256
  Coherence (1-5)              4.8250


## 10. Combined Dashboard & Error Analysis

In [ ]:
# --- Combined Dashboard ---
print("╔" + "═"*64 + "╗")
print("║" + f"  RAG EVALUATION DASHBOARD — {EXPERIMENT_NAME}".center(64) + "║")
print("║" + f"  {RUN_TIMESTAMP}".center(64) + "║")
print("╠" + "═"*64 + "╣")
print("║" + "  CONFIGURATION".center(64) + "║")
print("╠" + "═"*64 + "╣")
for k, v in [("Embedding", EMBEDDING_MODEL), ("Generator", GENERATOR_LLM),
             ("Search", f"{SEARCH_TYPE} top-{TOP_K}"), ("Chunks", f"{CHUNK_SIZE}/{CHUNK_OVERLAP}")]:
    print(f"║  {k:<18s} {str(v):<43s}".ljust(65) + "║")
print("╠" + "═"*64 + "╣")
print("║" + "  RETRIEVAL METRICS".center(64) + "║")
print("╠" + "═"*64 + "╣")
for name, val in retrieval_metrics.items():
    s = "✅" if val >= 0.6 else "⚠️" if val >= 0.4 else "❌"
    print(f"║  {s} {name:<22s} {val:.4f}".ljust(65) + "║")
print("╠" + "═"*64 + "╣")
print("║" + "  GENERATION METRICS".center(64) + "║")
print("╠" + "═"*64 + "╣")
for name, (val, warn, good) in [
    ("Answer Relevance", (generation_metrics["Answer Relevance (1-5)"], 3.5, 4.0)),
    ("Groundedness", (generation_metrics["Groundedness (1-5)"], 3.5, 4.0)),
    ("Hallucination Rate", (generation_metrics["Hallucination Rate (0-1)"], 0.20, 0.10)),
    ("Coherence", (generation_metrics["Coherence (1-5)"], 3.5, 4.0)),
]:
    s = ("✅" if val >= good else "⚠️" if val >= warn else "❌") if "Halluc" not in name else \
        ("✅" if val <= good else "⚠️" if val <= warn else "❌")
    print(f"║  {s} {name:<22s} {val:.4f}".ljust(65) + "║")
print("╚" + "═"*64 + "╝")

╔════════════════════════════════════════════════════════════════╗
║              RAG EVALUATION DASHBOARD — baseline_v1            ║
║                       2026-03-18 13:03:21                      ║
╠════════════════════════════════════════════════════════════════╣
║                          CONFIGURATION                         ║
╠════════════════════════════════════════════════════════════════╣
║  Embedding          text-embedding-3-small                     ║
║  Generator          gpt-4.1-mini                               ║
║  Search             similarity top-5                           ║
║  Chunks             3500/500                                   ║
╠════════════════════════════════════════════════════════════════╣
║                        RETRIEVAL METRICS                       ║
╠════════════════════════════════════════════════════════════════╣
║  ✅ Hit Rate@5             1.0000                               ║
║  ✅ MRR@5                  0.9271                            

In [ ]:
# --- Error Analysis: flag problematic queries ---
print("\n🔍 PROBLEMATIC QUERIES")
print("=" * 95)
print(f"  {'ID':<8s} {'Hit':>4s} {'Rel':>5s} {'Grd':>5s} {'Hal':>6s}  Diagnosis")
print("-" * 95)

flagged = 0
for r in e2e_results:
    issues = []
    if r["retrieval_hit"] == 0: issues.append("❌ Retrieval miss")
    if 0 < r["groundedness"] <= 2: issues.append("⚠️ Poorly grounded")
    if r["hallucination_rate"] > 0.3: issues.append("🔴 High hallucination")
    if 0 < r["answer_relevance"] <= 2: issues.append("⚠️ Low relevance")

    if issues:
        flagged += 1
        print(f"  {r['question_id']:<8s} {'❌' if r['retrieval_hit']==0 else '✅':>4s} "
              f"{r['answer_relevance']:>5d} {r['groundedness']:>5d} {r['hallucination_rate']:>6.2f}  "
              f"{' | '.join(issues)}")

if flagged == 0:
    print("  ✅ No problematic queries found!")
else:
    print(f"\n  Total flagged: {flagged}/{len(e2e_results)} queries")


🔍 PROBLEMATIC QUERIES
  ID        Hit   Rel   Grd    Hal  Diagnosis
-----------------------------------------------------------------------------------------------
  att_01      ✅     5     3   0.33  🔴 High hallucination
  att_05      ✅     4     1   0.75  ⚠️ Poorly grounded | 🔴 High hallucination
  att_08      ✅     5     3   0.50  🔴 High hallucination
  att_09      ✅     4     2   0.67  ⚠️ Poorly grounded | 🔴 High hallucination
  res_01      ✅     5     3   0.43  🔴 High hallucination
  res_03      ✅     5     3   0.31  🔴 High hallucination
  res_07      ✅     5     2   0.75  ⚠️ Poorly grounded | 🔴 High hallucination
  res_08      ✅     5     2   0.58  ⚠️ Poorly grounded | 🔴 High hallucination
  res_09      ✅     5     4   1.00  🔴 High hallucination
  res_10      ✅     5     1   1.00  ⚠️ Poorly grounded | 🔴 High hallucination
  cnn_01      ✅     5     4   0.40  🔴 High hallucination
  cnn_03      ✅     5     4   0.31  🔴 High hallucination
  cnn_04      ✅     5     3   0.36  🔴 High hal

## 11. Export All Results to Excel

All evaluation results — configuration snapshot, aggregate metrics, per-query retrieval details, per-query generation scores, and K-sweep data — are saved to a single Excel workbook with **multiple sheets** for easy cross-experiment comparison.

> **Tip:** Run the notebook with different configurations (change Section 3, hit Run All), and each run produces a separate Excel file named by `EXPERIMENT_NAME`. Compare sheets side-by-side in Excel or Google Sheets.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

wb = Workbook()

# ═══════════════════════════════════════════════════════════
# SHEET 1: Configuration + Summary
# ═══════════════════════════════════════════════════════════
ws_summary = wb.active
ws_summary.title = "Summary"

# Styling
header_font = Font(name="Arial", bold=True, size=12)
subheader_font = Font(name="Arial", bold=True, size=10)
value_font = Font(name="Arial", size=10)
header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
header_text = Font(name="Arial", bold=True, size=10, color="FFFFFF")
good_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
warn_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
bad_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

# Title
ws_summary.merge_cells('A1:D1')
ws_summary['A1'] = f"RAG Evaluation Report — {EXPERIMENT_NAME}"
ws_summary['A1'].font = Font(name="Arial", bold=True, size=14)
ws_summary['A2'] = f"Run: {RUN_TIMESTAMP}"
ws_summary['A2'].font = value_font

# Configuration section
row = 4
ws_summary.cell(row=row, column=1, value="CONFIGURATION").font = header_font
row += 1
config_data = [
    ("Experiment Name", EXPERIMENT_NAME),
    ("Experiment Notes", EXPERIMENT_NOTES),
    ("Embedding Model", EMBEDDING_MODEL),
    ("Generator LLM", GENERATOR_LLM),
    ("Generator Temperature", GENERATOR_TEMPERATURE),
    ("Judge LLM", JUDGE_LLM),
    ("Chunk Size (chars)", CHUNK_SIZE),
    ("Chunk Overlap (chars)", CHUNK_OVERLAP),
    ("Search Type", SEARCH_TYPE),
    ("Top-K", TOP_K),
    ("Relevance Threshold", RELEVANCE_THRESHOLD),
    ("Total Corpus Docs", len(total_docs)),
    # ("Wiki Docs", len(wiki_docs)),
    ("PDF Chunks", len(paper_docs)),
    ("Eval Dataset Size", len(eval_dataset)),
]
if SEARCH_TYPE == "mmr":
    config_data.extend([("MMR Fetch-K", MMR_FETCH_K), ("MMR Lambda", MMR_LAMBDA)])

for param, val in config_data:
    ws_summary.cell(row=row, column=1, value=param).font = subheader_font
    ws_summary.cell(row=row, column=2, value=str(val)).font = value_font
    row += 1

# Retrieval metrics
row += 1
ws_summary.cell(row=row, column=1, value="RETRIEVAL METRICS").font = header_font
row += 1
for name, val in retrieval_metrics.items():
    ws_summary.cell(row=row, column=1, value=name).font = value_font
    cell = ws_summary.cell(row=row, column=2, value=round(val, 4))
    cell.font = value_font
    cell.fill = good_fill if val >= 0.6 else warn_fill if val >= 0.4 else bad_fill
    row += 1

# Generation metrics
row += 1
ws_summary.cell(row=row, column=1, value="GENERATION METRICS").font = header_font
row += 1
for name, val in generation_metrics.items():
    ws_summary.cell(row=row, column=1, value=name).font = value_font
    cell = ws_summary.cell(row=row, column=2, value=round(val, 4))
    cell.font = value_font
    if "Halluc" in name:
        cell.fill = good_fill if val <= 0.10 else warn_fill if val <= 0.20 else bad_fill
    else:
        cell.fill = good_fill if val >= 4.0 else warn_fill if val >= 3.5 else bad_fill
    row += 1

# Column widths
ws_summary.column_dimensions['A'].width = 28
ws_summary.column_dimensions['B'].width = 50

# ═══════════════════════════════════════════════════════════
# SHEET 2: Per-Query Results (combined retrieval + generation)
# ═══════════════════════════════════════════════════════════
ws_queries = wb.create_sheet("Per-Query Results")

query_headers = ["Question ID", "Source Paper", "Question", "Retrieval Hit",
                 "Reciprocal Rank", "Precision@K",
                 "Answer Relevance", "Groundedness", "Hallucination Rate", "Coherence",
                 "RAG Answer (truncated)", "Relevance Reason", "Groundedness Reason"]

for col, header in enumerate(query_headers, 1):
    cell = ws_queries.cell(row=1, column=col, value=header)
    cell.font = header_text
    cell.fill = header_fill
    cell.border = thin_border

for row_idx, (pqr, e2e) in enumerate(zip(per_query_retrieval, e2e_results), start=2):
    values = [
        e2e["question_id"], e2e["source"], e2e["question"][:80],
        pqr["hit"], round(pqr["reciprocal_rank"], 3), round(pqr["precision_at_k"], 3),
        e2e["answer_relevance"], e2e["groundedness"],
        round(e2e["hallucination_rate"], 3) if e2e["hallucination_rate"] >= 0 else "N/A",
        e2e["coherence"],
        e2e.get("rag_answer", "")[:200],
        e2e.get("rel_reason", ""), e2e.get("grd_reason", ""),
    ]
    for col, val in enumerate(values, 1):
        cell = ws_queries.cell(row=row_idx, column=col, value=val)
        cell.font = value_font
        cell.border = thin_border
        # Color-code key columns
        if col == 4:  # Hit
            cell.fill = good_fill if val == 1 else bad_fill
        if col == 7 and isinstance(val, int):  # Relevance
            cell.fill = good_fill if val >= 4 else warn_fill if val >= 3 else bad_fill
        if col == 8 and isinstance(val, int):  # Groundedness
            cell.fill = good_fill if val >= 4 else warn_fill if val >= 3 else bad_fill
        if col == 9 and isinstance(val, (int, float)) and val >= 0:  # Hallucination
            cell.fill = good_fill if val <= 0.1 else warn_fill if val <= 0.3 else bad_fill

# Auto-adjust some widths
for col_letter, width in [('A', 10), ('B', 22), ('C', 45), ('K', 50), ('L', 40), ('M', 40)]:
    ws_queries.column_dimensions[col_letter].width = width

# ═══════════════════════════════════════════════════════════
# SHEET 3: K-Sweep Results
# ═══════════════════════════════════════════════════════════
ws_ksweep = wb.create_sheet("K-Sweep")

k_headers = ["K", "Hit Rate", "MRR", "Precision@K", "Recall@K", "nDCG@K"]
for col, header in enumerate(k_headers, 1):
    cell = ws_ksweep.cell(row=1, column=col, value=header)
    cell.font = header_text
    cell.fill = header_fill
    cell.border = thin_border

for row_idx, kr in enumerate(k_sweep_results, start=2):
    for col, val in enumerate([kr["k"], kr["hit_rate"], kr["mrr"],
                                kr["precision"], kr["recall"], kr["ndcg"]], 1):
        cell = ws_ksweep.cell(row=row_idx, column=col, value=round(val, 4) if isinstance(val, float) else val)
        cell.font = value_font
        cell.border = thin_border

# ═══════════════════════════════════════════════════════════
# SHEET 4: Per-Paper Breakdown
# ═══════════════════════════════════════════════════════════
ws_papers = wb.create_sheet("Per-Paper Breakdown")

paper_headers = ["Paper", "Hit Rate", "MRR", "Precision@K", "Recall@K", "nDCG@K"]
for col, header in enumerate(paper_headers, 1):
    cell = ws_papers.cell(row=1, column=col, value=header)
    cell.font = header_text
    cell.fill = header_fill

for row_idx, (paper, vals) in enumerate(sorted(paper_ret_detail.items()), start=2):
    ws_papers.cell(row=row_idx, column=1, value=paper).font = value_font
    for col, key in enumerate(["hit", "mrr", "prec", "rec", "ndcg"], 2):
        ws_papers.cell(row=row_idx, column=col, value=round(vals[key], 4)).font = value_font

ws_papers.column_dimensions['A'].width = 30

# ═══════════════════════════════════════════════════════════
# Save
# ═══════════════════════════════════════════════════════════
wb.save(EXCEL_OUTPUT_PATH)
print(f"✅ Results exported to: {EXCEL_OUTPUT_PATH}")
print(f"   Sheets: Summary | Per-Query Results | K-Sweep | Per-Paper Breakdown")

# Download the file
from google.colab import files
files.download(EXCEL_OUTPUT_PATH)

✅ Results exported to: rag_eval_results_baseline_v1.xlsx
   Sheets: Summary | Per-Query Results | K-Sweep | Per-Paper Breakdown


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Cost & Performance Estimates

In [ ]:
cost_estimates = {
    "Embedding (indexing corpus)": {"usd": 0.005, "note": f"~{len(total_docs)} chunks"},
    "Embedding (40 retrieval queries)": {"usd": 0.001, "note": "40 × ~20 tokens"},
    "RAG answers (40 calls)": {"usd": 0.025, "note": f"40 × {GENERATOR_LLM}"},
    "Judge — relevance (40)": {"usd": 0.015, "note": f"40 × {JUDGE_LLM}"},
    "Judge — groundedness (40)": {"usd": 0.025, "note": "includes context"},
    "Judge — hallucination (40)": {"usd": 0.025, "note": "includes context"},
    "Judge — coherence (40)": {"usd": 0.010, "note": "answer only"},
}

print("💰 ESTIMATED COST PER RUN")
print("=" * 65)
total_cost = 0
for comp, info in cost_estimates.items():
    print(f"  {comp:<38s} ${info['usd']:.3f}  ({info['note']})")
    total_cost += info["usd"]
print("-" * 65)
print(f"  {'TOTAL':<38s} ${total_cost:.3f}")

💰 ESTIMATED COST PER RUN
  Embedding (indexing corpus)            $0.005  (~79 chunks)
  Embedding (40 retrieval queries)       $0.001  (40 × ~20 tokens)
  RAG answers (40 calls)                 $0.025  (40 × gpt-4.1-mini)
  Judge — relevance (40)                 $0.015  (40 × gpt-5.2)
  Judge — groundedness (40)              $0.025  (includes context)
  Judge — hallucination (40)             $0.025  (includes context)
  Judge — coherence (40)                 $0.010  (answer only)
-----------------------------------------------------------------
  TOTAL                                  $0.106


---

## 13. Conclusion & Next Steps

### What We Covered

| Task | Key Takeaway |
|------|-------------|
| **Configuration Cell** | All RAG parameters in one place — change and re-run for instant A/B testing |
| **5 Retrieval Metrics** | Hit Rate, MRR, Precision@K, Recall@K, nDCG@K — pure Python implementations |
| **4 Generation Metrics** | Relevance, Groundedness, Hallucination Rate, Coherence — LLM-as-judge |
| **Per-Paper Breakdown** | Different documents have different retrieval/generation quality |
| **Error Analysis** | Pinpoint whether failures are in retrieval or generation |
| **K-Sweep** | Optimal K balances recall (higher K) vs precision (lower K) |
| **Excel Export** | Full results + config snapshot for cross-experiment comparison |

### Experiment Ideas — Change Config and Re-Run

| Change This | Expected Impact |
|-------------|----------------|
| `CHUNK_SIZE: 3500 → 1000` | More granular chunks → higher precision, possibly lower recall |
| `CHUNK_SIZE: 3500 → 5000` | Larger chunks → more context per chunk, but may reduce precision |
| `EMBEDDING_MODEL: small → large` | Higher-quality embeddings → better retrieval, but 6x cost |
| `SEARCH_TYPE: similarity → mmr` | Diverse chunks → better coverage, reduced redundancy |
| `TOP_K: 5 → 10` | More context for LLM → better recall, but more noise and token cost |
| `RELEVANCE_THRESHOLD: 0.3 → 0.2` | More lenient matching → higher metrics but potentially less meaningful |
| `GENERATOR_LLM: gpt-4.1-mini → gpt-5-mini` | Reasoning model → better answers, but no temp control |

### Try on Your Own

1. **A/B test chunk sizes:** Run with 1000, 2000, 3500, 5000 — compare Excel exports
2. **Compare embedding models:** `text-embedding-3-small` vs `text-embedding-3-large`
3. **Test hybrid search:** Change `SEARCH_TYPE` to `"mmr"` and tune `MMR_LAMBDA`
4. **Use semantic relevance matching:** Replace word overlap with embedding cosine similarity
5. **Add a reranker step:** Insert a cross-encoder between retrieval and generation
6. **Automate weekly runs:** Schedule via Colab + Google Sheets API for production monitoring

---

**Capstone Preview:** In the Capstone, you'll build a production RAG system on AWS with automated evaluation pipelines running on every knowledge base update. 🚀